<a href="https://colab.research.google.com/github/ayush4604/FitPulse-Health-Anomaly-Detection-from-Fitness-Devices/blob/main/Preprocess.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Mount Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    base_path = "/content/drive/MyDrive/FitPulse"
except ImportError:
    base_path = "." # Local fallback
    print("Local environment detected.")

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
import os

file_name = '/content/FitPulse_wearable_device_data.csv'
file_path = os.path.join(base_path, file_name)

if base_path == "." and not os.path.exists(file_path):
    file_path = file_name

print(f"Loading data from {file_path}...")
df = pd.read_csv(file_path)

Loading data from /content/FitPulse_wearable_device_data.csv...


In [3]:
print("Converting timestamps to UTC...")
if 'time_stamp' in df.columns:
    df['time_stamp'] = pd.to_datetime(df['time_stamp'], utc=True)
else:
    df['time_stamp'] = pd.to_datetime(df.filter(regex='(?i)time|date').iloc[:, 0], utc=True)

df.dropna(subset=['time_stamp'], inplace=True)

# standardize ID
df.rename(columns={'Patient_ID': 'Id', 'id': 'Id'}, inplace=True)
if 'Id' not in df.columns:
    df['Id'] = 'Default_User'
df['Id'] = df['Id'].astype(str)

Converting timestamps to UTC...


In [4]:
print("Aligning data to 1-minute intervals...")
processed_dfs = []

for user_id, user_df in df.groupby('Id'):
    user_df = user_df.sort_values('time_stamp')
    user_df.set_index('time_stamp', inplace=True)

    # Remove duplicates
    user_df = user_df[~user_df.index.duplicated(keep='first')]

    numeric_cols = user_df.select_dtypes(include=[np.number]).columns
    categorical_cols = user_df.select_dtypes(exclude=[np.number]).columns

    # Resample Numeric - Interpolate
    resampled_numeric = user_df[numeric_cols].resample('1min').mean()
    resampled_numeric = resampled_numeric.interpolate(method='time')

    # Resample Categorical - Forward Fill
    if len(categorical_cols) > 0:
        resampled_cat = user_df[categorical_cols].resample('1min').ffill()
        user_resampled = pd.concat([resampled_numeric, resampled_cat], axis=1)
    else:
        user_resampled = resampled_numeric

    # Cleanup NAs
    user_resampled.bfill(inplace=True)
    user_resampled.ffill(inplace=True)

    user_resampled['Id'] = user_id
    processed_dfs.append(user_resampled)

if processed_dfs:
    final_df = pd.concat(processed_dfs)
    final_df.reset_index(inplace=True)

    # Reorder columns: Id, time_stamp, then everything else
    cols = ['Id', 'time_stamp'] + [c for c in final_df.columns if c not in ['Id', 'time_stamp']]
    final_df = final_df[cols]

    print(f"Final Data Shape: {final_df.shape}")
    display(final_df.head())
else:
    print("No data processed.")


Aligning data to 1-minute intervals...
Final Data Shape: (7908, 7)


,Id,time_stamp,heart_rate,step_count,Weight,Height,sleep_tracking
0,P0001,2025-01-14 16:50:00+00:00,74.499653,26.0,88.0,168.0,awake
1,P0002,2025-01-21 08:55:00+00:00,90.833116,6.0,78.0,162.0,awake
2,P0003,2025-01-17 07:10:00+00:00,61.270774,40.0,64.0,162.0,awake
3,P0005,2025-01-18 18:25:00+00:00,59.807869,29.0,70.0,151.0,exercise
4,P0006,2025-01-13 07:20:00+00:00,92.702179,34.0,88.0,159.0,awake


In [7]:
output_file = 'cleaned_fitness_data.csv'
output_path = os.path.join(base_path, output_file)

if base_path == ".":
    output_path = output_file

final_df.to_csv(output_path, index=False)
print(f"Saved aligned data to {output_path}")
print(final_df.head())

Saved aligned data to /content/drive/MyDrive/FitPulse/cleaned_fitness_data.csv
      Id                time_stamp  heart_rate  step_count  Weight  Height  \
0  P0001 2025-01-14 16:50:00+00:00   74.499653        26.0    88.0   168.0   
1  P0002 2025-01-21 08:55:00+00:00   90.833116         6.0    78.0   162.0   
2  P0003 2025-01-17 07:10:00+00:00   61.270774        40.0    64.0   162.0   
3  P0005 2025-01-18 18:25:00+00:00   59.807869        29.0    70.0   151.0   
4  P0006 2025-01-13 07:20:00+00:00   92.702179        34.0    88.0   159.0   

  sleep_tracking  
0          awake  
1          awake  
2          awake  
3       exercise  
4          awake  
